# 第23章：截面研究

从并列值与缺失值开始，比较Pearson IC、Rank IC、池化与逐时点估计，并核对权重、换手、z/N近似和Fama–MacBeth。随后生成300项资产、180个月的构造面板，观察重叠标签的方差膨胀。French资料属于月频历史描述，默认不执行该分支或展示其图库；本例不新写结果文件。

[本语言逐本运行模式与准备条件](../NOTEBOOK_GUIDE.md)。本阶段只制作语言层，便携环境与执行在后续隔离冷启动核验。

In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch23_cross_sectional_research_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

import sys
from pathlib import Path

sys.path.insert(0, str(ROOT / "src"))
print("interpreter:", sys.executable)
print("project root:", ROOT)
import importlib.util


def load_code_module(filename: str):
    """Import a chapter CLI from code/ so the notebook calls exactly the functions the CLI runs."""
    spec = importlib.util.spec_from_file_location(Path(filename).stem, ROOT / "code" / filename)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

## 1. 名次、百分位、分位桶（8.1–8.2）

In [ ]:
import numpy as np
import pandas as pd
from quant_course.factor_research import percentile_cross_section, quantile_bucket, rank_cross_section

x = np.array([3, 1, np.nan, 3, 2, 7, 5, 5])
print("rank      ", rank_cross_section(x))
print("percentile", percentile_cross_section(x).round(4))
np.testing.assert_allclose(rank_cross_section(x), [3.5, 1, np.nan, 3.5, 2, 7, 5.5, 5.5])
np.testing.assert_array_equal(quantile_bucket(np.arange(1.0, 11.0), 5), [1, 1, 2, 2, 3, 3, 4, 4, 5, 5])
np.testing.assert_array_equal(quantile_bucket([1, 2, 2, 2, 5, 6, 7, 8], 4), [1, 2, 2, 2, 3, 3, 4, 4])

## 2. 离群值下的 Pearson 与 Rank IC（8.3）；混合相关与逐日 IC（8.4）

In [ ]:
from quant_course.factor_research import ic_series, information_coefficient

s = np.array([1, 2, 3, 4, 5, 20.0]); y = np.array([0.02, 0.01, 0.04, 0.03, 0.06, -0.05])
print("Pearson", round(information_coefficient(s, y, method="pearson"), 4), "| Spearman", round(information_coefficient(s, y, method="spearman"), 4))
idx = pd.period_range("2020-01", periods=2, freq="M")
signal = pd.DataFrame([[1, 2, 3, 4.0], [11, 12, 13, 14.0]], index=idx, columns=list("abcd"))
label = pd.DataFrame([[0.10, 0.12, 0.14, 0.16], [0.26, 0.24, 0.22, 0.20]], index=idx, columns=list("abcd"))
per_date = ic_series(signal, label, method="pearson")["ic"]
pooled = np.corrcoef(signal.to_numpy().ravel(), label.to_numpy().ravel())[0, 1]
print("per-date IC", per_date.round(3).tolist(), "pooled", round(pooled, 4))
assert abs(pooled - 0.89087) < 1e-5 and abs(per_date.mean()) < 1e-12

## 3. 五分位多空、权重、换手（8.5）

In [ ]:
from quant_course.factor_research import bucket_membership_turnover, bucket_returns, quantile_long_short_weights, target_turnover

cols = list("abcdefghij")
signal = pd.DataFrame([[5, 3, 9, 1, 7, 2, 10, 4, 8, 6.0], [5, 4, 9, 1, 6, 3, 10, 2, 7, 8.0]], index=idx, columns=cols)
label = pd.DataFrame([[.02, .01, .05, -.01, .03, .00, .06, .01, .04, .02], [.01, .02, .03, -.02, .02, .01, .03, .00, .04, .05]], index=idx, columns=cols)
buckets = bucket_returns(signal, label, 5)
w = quantile_long_short_weights(signal, 5)
turnover = target_turnover(w)
assert np.allclose(buckets["LS"], [0.06, 0.04]) and abs(turnover.iloc[1] - 1.0) < 1e-12
print(buckets.round(3)); print("target turnover", turnover.iloc[1], "| bottom-bucket membership turnover", bucket_membership_turnover(signal, 5, bucket=1).iloc[1])

## 4. $z/N$ 组合收益 = IC × σ_y（8.6）与 Fama–MacBeth 三期（8.7）

In [ ]:
from quant_course.factor_research import fama_macbeth

s6 = np.array([2, 4, 6, 8, 10.0]); y6 = np.array([0.01, 0.00, 0.03, 0.02, 0.04])
z6 = (s6 - s6.mean()) / s6.std(ddof=0)
print("R_p =", round((z6 * y6).sum() / 5, 6), "| IC*sd0(y) =", round(np.corrcoef(s6, y6)[0, 1] * y6.std(ddof=0), 6))
idx3 = pd.period_range("2020-01", periods=3, freq="M")
sig3 = pd.DataFrame([[1, 2, 3, 4.0], [4, 3, 2, 1.0], [1, 3, 2, 4.0]], index=idx3, columns=list("abcd"))
lab3 = pd.DataFrame([[.01, .02, .04, .05], [.02, .02, .01, .00], [.00, .03, .02, .04]], index=idx3, columns=list("abcd"))
fm = fama_macbeth(lab3, {"signal": sig3}, min_extra_obs=0)
print(fm.slopes["signal"].round(4).tolist(), fm.summary.loc["signal"].round(5).to_dict())
assert abs(fm.summary.loc["signal", "mean"] - 0.011333333) < 1e-8

## 5. 构造面板（N=300，T=180）：IC、衰减、分位、恒等式

In [ ]:
ch23 = load_code_module("ch23_cross_sectional_research.py")
panel = ch23.constructed_panel()
study = ch23.constructed_study(panel)
print("Spearman IC mean", round(study["ic_spearman"]["mean"], 4), "t", round(study["ic_spearman"]["t_naive"], 1), "| hit rate", round(study["ic_spearman"]["hit_rate"], 3))
print("decile LS %/mo", round(study["decile_ls_top_minus_bottom"]["mean_pct"], 3), "| quintile EW", round(study["quintile_ls_equal_weight"]["mean_pct"], 3),
      "| VW", round(study["quintile_ls_value_weight"]["mean_pct"], 3), "| proportional", round(study["proportional_gross2_ls"]["mean_pct"], 3))
assert study["identity_z_over_n_vs_ic_times_sd0_max_gap"] < 1e-12
decay = pd.DataFrame(study["ic_decay_spearman"]).T
decay.round(4)

## 6. 混合相关 vs 逐日 IC；重叠标签的方差膨胀因子

In [ ]:
demo = ch23.pooled_vs_per_date()
print("pooled", round(demo["pooled_corr"], 3), "mean per-date IC", round(demo["mean_ic"], 4))
assert demo["pooled_corr"] > 0.2 > abs(demo["mean_ic"])
vif = pd.DataFrame({h: ch23.overlap_vif(0.85, h) for h in (1, 3, 6, 12)}).T
print(vif.round(3))
assert abs(vif.loc[3, "vif_true"] - 2.615) < 1e-3

## 7. French月度行业分析（默认跳过）

原“逐日”在此历史月频输入下应为逐月；不构造逐股PIT信号，不产生本次历史输出。

In [ ]:
if RUN_FRENCH:
    from quant_course.french_frozen import load_industries_and_rf
    returns, _, prov = load_industries_and_rf()
    real = ch23.industry_study(returns)
    by = pd.DataFrame(real["rank_ic_by_period"]).T[["mean", "t_nw", "n_dates"]]
    print("French is point-in-time:", prov["is_point_in_time"]); print(by.round(4))
    assert real["months_with_ic"] == 1190 and abs(by.loc["2000-01..2026-07", "t_nw"] - 1.6993) < 1e-3
    print("quintile means %/mo:", {k: round(v, 3) for k, v in real["quintile_mean_return_pct"].items()}, "| LS", round(real["quintile_ls_mean_pct"], 3))
    print("FM slope on z (mean, t_nw):", round(real["fama_macbeth_signal_z"]["signal_z"]["mean"], 6), round(real["fama_macbeth_signal_z"]["signal_z"]["t_nw"], 2))
    assert real["fm_slope_vs_ic_times_sd0_gap"] < 1e-12
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


In [ ]:
print("SKIPPED historical figure gallery; constructed calculations above are freshly computed; French historical images are not reproduced")


## 8. 自己试一试

1. 把第 3 节的分位数改为 $q=10$，观察 `quantile_bucket` 在 $N=10$ 时每桶只有 1 个资产的含义。
2. 在第 6 节把信号自相关改为 0，用 `overlap_vif(0, 3)` 验证 VIF=1。
3. 把 French 信号窗口改为 `formation_return(returns, 1, 0)`（上月收益），比较 IC 与衰减曲线。

涉及French或industry_study的题目保留原研究设计问题，当前只讨论如何改设计及解释结果。市场分支默认禁用，G01权利unknown且G05/A10-B执行未授权；本轮不运行这些历史实验。其余构造练习沿本Notebook实际单元使用。